# Phase 1 — Data Ingestion

Downloads and merges all three data sources into a single `master.parquet`.

**Run this notebook first** — every other notebook depends on its output.

### Install dependencies
```bash
pip install pandas pyarrow requests tqdm meteostat holidays
```

### Data sources
| Source | What | License |
|--------|------|---------|
| OPSD | German load + solar + wind (hourly) | MIT / CC BY 4.0 |
| SMARD | Official grid validation data | dl-de/by-2-0 |
| Meteostat | Weather (5 DE stations) | CC BY-NC 4.0 |

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')
RESULTS   = Path('../results')

for d in [RAW, PROCESSED, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 110
print('Directories ready:')
for d in [RAW, PROCESSED, RESULTS]:
    print(f'  {d.resolve()}')

## 1. OPSD — Primary dataset

Downloads ~150 MB CSV from the OPSD data portal. Cached after first run.

In [ ]:
from src.ingest.opsd import load_opsd

opsd = load_opsd(RAW, PROCESSED, force=False)
print(f'Shape: {opsd.shape}')
print(f'Range: {opsd.index.min().date()} -> {opsd.index.max().date()}')
print(f'\nMissing values (%):')
print((opsd.isna().mean() * 100).round(2).to_string())
opsd.tail(3)

In [ ]:
# Quick sanity plot
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
cols = ['load_mw', 'solar_mw', 'wind_onshore_mw', 'wind_offshore_mw']

for ax, col in zip(axes, cols):
    opsd[col].resample('W').mean().plot(ax=ax, lw=0.8)
    ax.set_title(col)
    ax.set_ylabel('MW (weekly avg)')

axes[-1].set_xlabel('Date')
plt.suptitle('OPSD — German electricity data overview', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'opsd_overview.png', bbox_inches='tight')
plt.show()
print(f'Saved -> {RESULTS}/opsd_overview.png')

## 2. SMARD — Validation dataset

Fetches from the official SMARD API (Bundesnetzagentur).
This can take **5–15 minutes** on first run (many weekly chunks to fetch).
All chunks are cached in `data/raw/` as JSON files — subsequent runs are instant.

Set `START` to a more recent date (e.g. `'2020-01-01'`) to reduce download time.

In [ ]:
from src.ingest.smard import load_smard

START = '2018-01-01'   # change to '2020-01-01' for a faster first run

smard = load_smard(RAW, PROCESSED, start=START, force=False)
print(f'Shape: {smard.shape}')
print(f'Range: {smard.index.min().date()} -> {smard.index.max().date()}')
print(f'\nMissing values (%):')
print((smard.isna().mean() * 100).round(2).to_string())
smard.tail(3)

In [ ]:
# Spot-check: OPSD vs SMARD load — should track closely
fig, ax = plt.subplots(figsize=(14, 4))
overlap_start = max(opsd.index.min(), smard.index.min())
sample = slice(str(overlap_start.date()), str((overlap_start + pd.Timedelta(days=30)).date()))

opsd['load_mw'][sample].plot(ax=ax, label='OPSD load_mw',      lw=1.5, color='steelblue')
smard['load_mwh'][sample].plot(ax=ax, label='SMARD load_mwh',  lw=1.2, color='crimson', alpha=0.7, ls='--')
ax.set_title('OPSD vs SMARD load — 30-day cross-check')
ax.set_ylabel('MW / MWh')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / 'opsd_vs_smard_crosscheck.png', bbox_inches='tight')
plt.show()

## 3. Meteostat — Weather data

Downloads hourly weather for 5 German stations via Meteostat v2.
Uses direct bulk CSV download as fallback if the library call fails.

First run: ~2–5 minutes. Cached to `data/processed/meteostat.parquet` after that.

In [ ]:
from src.ingest.meteostat import load_meteostat

weather = load_meteostat(PROCESSED, start=START, force=False)
print(f'Shape: {weather.shape}')
print(f'Range: {weather.index.min().date()} -> {weather.index.max().date()}')

# Show composite columns only
composite_cols = [c for c in weather.columns if c.startswith('de_')]
print(f'\nNational composite columns: {composite_cols}')
print(f'\nMissing values (composite columns, %):')
print((weather[composite_cols].isna().mean() * 100).round(2).to_string())
weather[composite_cols].tail(3)

In [ ]:
# Weather overview plot
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
sample_weather = weather[composite_cols].resample('D').mean()

sample_weather['de_temp'].plot(ax=axes[0], color='crimson', lw=0.8)
axes[0].set_title('National avg temperature (°C)')

sample_weather['de_wspd'].plot(ax=axes[1], color='steelblue', lw=0.8)
axes[1].set_title('National avg wind speed (km/h)')

sample_weather['de_tsun'].plot(ax=axes[2], color='goldenrod', lw=0.8)
axes[2].set_title('National avg sunshine (min/h)')

for ax in axes:
    ax.set_xlabel('')
axes[-1].set_xlabel('Date')

plt.suptitle('Meteostat — German national weather composites (daily avg)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'meteostat_overview.png', bbox_inches='tight')
plt.show()

## 4. Merge → master.parquet

Aligns all three sources to a clean hourly UTC index and saves `master.parquet`.

In [ ]:
from src.ingest.merge import build_master

master = build_master(
    raw_dir=RAW,
    processed_dir=PROCESSED,
    start=START,
    force=False,
)

print(f'Master shape : {master.shape}')
print(f'Date range   : {master.index.min().date()} -> {master.index.max().date()}')
print(f'\nAll columns:')
for col in master.columns:
    nan_pct = master[col].isna().mean() * 100
    print(f'  {col:<35}  {nan_pct:5.1f}% NaN')

## 5. EDA — Data quality and correlation

Quick checks before handing off to Phase 2.

In [ ]:
# Gap heatmap — missing values by month
target_cols = ['load_mw', 'solar_mw', 'wind_onshore_mw', 'wind_offshore_mw']
monthly_nan = (
    master[target_cols]
    .resample('ME')
    .apply(lambda x: x.isna().mean() * 100)
)

import seaborn as sns
fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(monthly_nan.T, ax=ax, cmap='Reds', cbar_kws={'label': 'NaN %'},
            linewidths=0.3, linecolor='white')
ax.set_title('Missing data (%) by month — target columns')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig(RESULTS / 'missing_data_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation: weather vs generation targets
weather_composite = [c for c in master.columns if c.startswith('de_')]
corr_cols = target_cols + weather_composite
corr = master[corr_cols].corr()[target_cols].drop(target_cols)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Pearson correlation: weather features vs targets')
plt.tight_layout()
plt.savefig(RESULTS / 'weather_target_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Descriptive statistics
print('=== Target column statistics (MW) ===')
print(master[target_cols].describe().round(1).to_string())
print(f'\nmaster.parquet saved to: {(PROCESSED / "master.parquet").resolve()}')
print('\nPhase 1 complete. Run 02_feature_engineering.ipynb next.')